In [1]:
dataset_url="https://huggingface.co/datasets/SetFit/bbc-news"

In [2]:
!pip install datasets
!pip install transformers
!pip install accelerate
!pip install evaluate
!pip install sentence-transformers

In [3]:
from datasets import load_dataset

ds = load_dataset("SetFit/bbc-news")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from pprint import pprint

# Get a sample of the dataset
sample = ds['train'].select(range(2))

# Print the sample
for i, item in enumerate(sample):
    print(f"Sample {i+1}:")
    pprint(item)
    print("-" * 40)

Sample 1:
{'label': 2,
 'label_text': 'sport',
 'text': 'wales want rugby league training wales could follow england s lead '
         'by training with a rugby league club.  england have already had a '
         'three-day session with leeds rhinos  and wales are thought to be '
         'interested in a similar clinic with rivals st helens. saints coach '
         'ian millward has given his approval  but if it does happen it is '
         'unlikely to be this season. saints have a week s training in '
         'portugal next week  while wales will play england in the opening six '
         'nations match on 5 february.  we have had an approach from wales   '
         'confirmed a saints spokesman.  it s in the very early stages but it '
         'is something we are giving serious consideration to.  st helens  who '
         'are proud of their welsh connections  are obvious partners for the '
         'welsh rugby union  despite a spat in 2001 over the collapse of '
         'kiero

In [5]:
# Get all unique label_text values from the train split
unique_categories = set(ds['train']['label_text'])

# Print the unique categories
print("Unique categories:")
for category in sorted(unique_categories):
    print(f"- {category}")

Unique categories:
- business
- entertainment
- politics
- sport
- tech


In [6]:
df = ds['train'].to_pandas()
# count number of row for each label text
df.groupby('label_text').count()

,text,label
label_text,,
business,286,286
entertainment,210,210
politics,242,242
sport,275,275
tech,212,212


In [7]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForMaskedLM.from_pretrained(model_id)

text = "The capital of France is [MASK]."
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)

# To get predictions for the mask:
masked_index = inputs["input_ids"][0].tolist().index(tokenizer.mask_token_id)
predicted_token_id = outputs.logits[0, masked_index].argmax(axis=-1)
predicted_token = tokenizer.decode(predicted_token_id)
print("Predicted token:", predicted_token)

Predicted token:  Paris


In [8]:
from transformers import AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)
model.eval()

ModernBertModel(
  (embeddings): ModernBertEmbeddings(
    (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (layers): ModuleList(
    (0): ModernBertEncoderLayer(
      (attn_norm): Identity()
      (attn): ModernBertAttention(
        (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
        (rotary_emb): ModernBertRotaryEmbedding()
        (Wo): Linear(in_features=768, out_features=768, bias=False)
        (out_drop): Identity()
      )
      (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): ModernBertMLP(
        (Wi): Linear(in_features=768, out_features=2304, bias=False)
        (act): GELUActivation()
        (drop): Dropout(p=0.0, inplace=False)
        (Wo): Linear(in_features=1152, out_features=768, bias=False)
      )
    )
    (1-21): 21 x ModernBertEncoderLayer(
      (attn_norm): LayerNorm((768,), eps=1e-05, e

In [9]:
text = "The capital of France is Paris."
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
attention_mask = inputs['attention_mask']

with torch.no_grad():
    outputs = model(**inputs)
    last_hidden_state = outputs.last_hidden_state

print(inputs)
print("==========")
decoded_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(decoded_tokens)
print("==========")
print(last_hidden_state.shape)
print("==========")
print(last_hidden_state[0][0][:50])

{'input_ids': tensor([[50281,   510,  5347,   273,  6181,   310,  7785,    15, 50282]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}
['[CLS]', 'The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis', 'ĠParis', '.', '[SEP]']
torch.Size([1, 9, 768])
tensor([ 0.2510, -0.8900, -0.7447, -0.1338, -0.6758, -0.6268, -1.0714, -1.1531,
         0.5290, -1.0357,  0.0034,  0.5587, -1.3051, -0.1262, -0.5455, -0.3348,
        -0.3254,  0.6492,  0.5990,  1.0442, -0.4831, -0.4719, -0.1171, -0.0445,
        -0.5067, -0.6600, -1.2544,  0.1157, -0.2932,  0.7848, -1.2969,  0.6125,
        -0.1215, -0.2493,  0.2903,  0.5177, -0.7596, -0.7141, -0.3428, -0.3654,
        -0.0329, -0.2163,  0.5728, -0.5759,  0.1778,  0.4421,  1.1398, -1.0408,
        -0.5048,  0.2062])


In [10]:
text = "The capital of India is Mumbai."
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
attention_mask = inputs['attention_mask']

with torch.no_grad():
    outputs = model(**inputs)
    last_hidden_state = outputs.last_hidden_state
print(inputs)
print("==========")
decoded_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(decoded_tokens)
print("==========")
print(last_hidden_state.shape)
print("==========")
print(last_hidden_state[0][0][:50])

{'input_ids': tensor([[50281,   510,  5347,   273,  5427,   310, 32111,    15, 50282]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}
['[CLS]', 'The', 'Ġcapital', 'Ġof', 'ĠIndia', 'Ġis', 'ĠMumbai', '.', '[SEP]']
torch.Size([1, 9, 768])
tensor([ 0.3359, -0.9154, -0.7217,  0.0922, -0.6907, -0.7731, -0.9339, -1.0410,
         0.8516, -0.8657, -0.3942,  0.6186, -1.4260, -0.0695, -0.3043, -0.4688,
        -0.2930,  0.6086,  0.4719,  0.9484, -0.3669, -0.3670, -0.0173, -0.0379,
        -0.4910, -0.5951, -1.1534,  0.0533, -0.0938,  0.7070, -1.3225,  0.3698,
        -0.2744, -0.3663,  0.2587,  0.3546, -0.6846, -0.7215, -0.3908, -0.6234,
         0.0127, -0.2473,  0.6476, -0.5683,  0.0760,  0.6380,  0.9180, -0.9159,
        -0.4755,  0.2118])


In [11]:
# as first token embedding represent entire sentense meaning selecting only first
last_hidden_state[:, 0, :].shape

torch.Size([1, 768])

In [12]:
input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
mean_pooled = sum_embeddings / sum_mask

In [13]:
print(attention_mask)
print(input_mask_expanded.shape)
print(input_mask_expanded)

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])
torch.Size([1, 9, 768])
tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]]])


In [14]:
print(sum_embeddings.shape)
print(sum_embeddings[0][:50])

torch.Size([1, 768])
tensor([  1.8829,  -6.7077,  -3.8820,   4.3029,   0.2900,  -2.6849,   2.8007,
         -0.1233,   4.1835,  -5.4224,  -7.1855,   1.7179,  -7.0962,   3.8826,
          0.0311,  -1.2560,  -1.0083,   1.5914,   4.7078,   1.3579,   2.1411,
          2.2440,   1.7104,   1.6066,  -3.8505,   3.8517,  -5.8592,  -2.7366,
          3.8252,   3.7523,  -0.1151, -18.2701,   4.9401,  -1.9201,   2.2748,
         -1.4707,  -4.5392,   1.0468,  -1.0625,  -5.4984,   0.2046,  -1.7916,
         -1.0935,   1.0443,   2.4692,  -1.1411,   0.0977,   3.9703,  -0.2278,
          2.5104])


In [15]:
print(sum_mask.shape)
print(sum_mask[0][:50])

torch.Size([1, 768])
tensor([9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9.,
        9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9.,
        9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9.])


In [16]:
print(mean_pooled.shape)
print(mean_pooled[0][:50])

torch.Size([1, 768])
tensor([ 0.2092, -0.7453, -0.4313,  0.4781,  0.0322, -0.2983,  0.3112, -0.0137,
         0.4648, -0.6025, -0.7984,  0.1909, -0.7885,  0.4314,  0.0035, -0.1396,
        -0.1120,  0.1768,  0.5231,  0.1509,  0.2379,  0.2493,  0.1900,  0.1785,
        -0.4278,  0.4280, -0.6510, -0.3041,  0.4250,  0.4169, -0.0128, -2.0300,
         0.5489, -0.2133,  0.2528, -0.1634, -0.5044,  0.1163, -0.1181, -0.6109,
         0.0227, -0.1991, -0.1215,  0.1160,  0.2744, -0.1268,  0.0109,  0.4411,
        -0.0253,  0.2789])


In [17]:
embedding = (last_hidden_state * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1, keepdim=True)

embedding_np = embedding
print(embedding_np.shape)
print(embedding_np[0][:50])

torch.Size([1, 768])
tensor([ 0.2092, -0.7453, -0.4313,  0.4781,  0.0322, -0.2983,  0.3112, -0.0137,
         0.4648, -0.6025, -0.7984,  0.1909, -0.7885,  0.4314,  0.0035, -0.1396,
        -0.1120,  0.1768,  0.5231,  0.1509,  0.2379,  0.2493,  0.1900,  0.1785,
        -0.4278,  0.4280, -0.6510, -0.3041,  0.4250,  0.4169, -0.0128, -2.0300,
         0.5489, -0.2133,  0.2528, -0.1634, -0.5044,  0.1163, -0.1181, -0.6109,
         0.0227, -0.1991, -0.1215,  0.1160,  0.2744, -0.1268,  0.0109,  0.4411,
        -0.0253,  0.2789])


In [18]:
from sentence_transformers import SentenceTransformer
sentence_model = SentenceTransformer(model_id)

sen_text = [text]
sen_embeddings = sentence_model.encode(sen_text)
print(sen_embeddings.shape)
print(sen_embeddings[0][:50])

(1, 768)
[ 0.20920736 -0.74530387 -0.43132842  0.4781012   0.03221883 -0.2983187
  0.31118664 -0.01369894  0.46483088 -0.60248446 -0.79838586  0.19087301
 -0.7884705   0.43139902  0.00345963 -0.13955756 -0.11203475  0.1768216
  0.5230847   0.15088247  0.23790519  0.2493301   0.19004901  0.17850997
 -0.42783883  0.42796624 -0.651017   -0.30406308  0.42502823  0.41692558
 -0.01278764 -2.0300152   0.54890037 -0.21334738  0.2527558  -0.163411
 -0.5043534   0.11631384 -0.11805063 -0.6109356   0.02273277 -0.19906528
 -0.12150526  0.11603837  0.27435222 -0.12678699  0.01085153  0.44114786
 -0.02531303  0.27893168]


In [ ]:
!pip install datasets
!pip install transformers
!pip install accelerate
!pip install evaluate


In [ ]:
dataset_url="https://huggingface.co/datasets/SetFit/bbc-news"
from datasets import load_dataset

ds = load_dataset("SetFit/bbc-news")


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset("SetFit/bbc-news")
model_id = "distilbert/distilbert-base-uncased" #"answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [ ]:
# Load F1 metric
import evaluate
import numpy as np


f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    f1_result = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    accuracy_result = accuracy_metric.compute(predictions=predictions, references=labels)

    return {
        "f1": f1_result["f1"],
        "accuracy": accuracy_result["accuracy"]
    }


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import AutoTokenizer, DataCollatorWithPadding

# Tokenize dataset
def preprocess(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

encoded_dataset = ds.map(preprocess, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Convert to pandas DataFrame (only needed columns)
df = ds['train'].to_pandas()[['label', 'label_text']]

# Drop duplicates to get unique (label, label_text) pairs
unique_label_map = df.drop_duplicates().sort_values('label')

# Build mappings
id2label = dict(zip(unique_label_map.label, unique_label_map.label_text))
label2id = {v: k for k, v in id2label.items()}


In [ ]:
import torch, gc

del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=len(label2id), id2label=id2label, label2id=label2id)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


cuda


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
!nvidia-smi

Sun Apr 20 15:39:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             30W /   70W |     364MiB /  15360MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Training arguments
training_args = TrainingArguments(
    run_name="distilbert/distilbert-base-uncased",
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    logging_steps=10,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to=[],
    fp16=True
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

# Evaluate
trainer.evaluate()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.042700,0.141094,0.966400,0.967000
2,0.169400,0.106612,0.974438,0.975000
3,0.004600,0.107366,0.978725,0.979000


{'eval_loss': 0.10736613720655441,
 'eval_f1': 0.9787251858846592,
 'eval_accuracy': 0.979,
 'eval_runtime': 5.3783,
 'eval_samples_per_second': 185.933,
 'eval_steps_per_second': 92.967,
 'epoch': 3.0}

In [ ]:
from datasets import Dataset

# Create a dataset with your single input
input_text = ["hitachi won electric top company award"]
inputs = tokenizer(input_text, truncation=True, padding=True, return_tensors="pt")
# Convert to Hugging Face Dataset format
predict_dataset = Dataset.from_dict({k: v.numpy() for k, v in inputs.items()})

# Run prediction
predictions = trainer.predict(predict_dataset)
predicted_scores = predictions.predictions.squeeze()
predicted_label = predicted_scores.argmax().item()
predicted_label_name = id2label[predicted_label]

print(f"Predicted label: {predicted_label_name}")

Predicted label: entertainment


In [ ]:
from transformers import pipeline
text_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer) #, return_all_scores=True)

Device set to use cuda:0


In [ ]:
# Run inference
input_text = "hitachi won electric top company award"
results = text_classifier(input_text)

print(results)

[{'label': 'entertainment', 'score': 0.950412929058075}]


In [ ]:
model.save_pretrained("distil_bert_for_classification")
tokenizer.save_pretrained("distil_bert_for_classification")


('distil_bert_for_classification/tokenizer_config.json',
 'distil_bert_for_classification/special_tokens_map.json',
 'distil_bert_for_classification/vocab.txt',
 'distil_bert_for_classification/added_tokens.json',
 'distil_bert_for_classification/tokenizer.json')

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# Load model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained("distil_bert_for_classification")
tokenizer = AutoTokenizer.from_pretrained("distil_bert_for_classification")

# Create a pipeline
text_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer) #, return_all_scores=True)

Device set to use cuda:0


In [ ]:
# Run inference
input_text = "hitachi won electric top company award"
results = text_classifier(input_text)

print(results)

[{'label': 'entertainment', 'score': 0.9506762027740479}]
